In [ ]:
### notebook config ###

table_path = None # where your metadata is (necessary)
output_folder = None # where to store your results
model_path = None # where to store your model
model_name = None # 'efficientnet_b0' or 'resnet18'
num_epochs = None
batch_size = None
num_workers = None # speeds up audio processing
freeze_inner = None # true means only update output layer weights
loss_name = None # 'cross_entropy' or 'bce_with_logits'
optimizer_name = None # 'adam' or 'sgd'
use_cosine_annealing = None # whether to use cosine annealing learning rate schedule
dropout = None # dropout rate for classifier head
lr = None # learning rate for optimizer

### AudioDataset override knobs ###

base_folder = None # where high-level acoustics data files lie
duration = None # seconds in clip
p_augment = None # probability of training data augmentation
num_augments = None # number of augmentation loops
mel_time_size = None # target time dimension size for mel spectrograms (applies random padding if specified)

### don't change these ###

n_samples = None # downsample for software dev purposes
min_gain_db = None
max_gain_db = None
fmin = None
fmax = None
sr = None # standardized sampling rate

# This needs to specify n_mels, etc

### Process the parameters

In [ ]:
# defaults
if fmax is None:
    from osprey import fmax
if fmin is None:
    from osprey import fmin
if duration is None:
    from osprey import duration
if base_folder is None:
    from osprey import base_folder
if sr is None:
    from osprey import sr
if p_augment is None:
    from osprey import p_augment
if min_gain_db is None:
    from osprey import min_gain_db
if max_gain_db is None:
    from osprey import max_gain_db
if num_augments is None:
    num_augments = 1
assert num_augments >= 1
num_augments = int(num_augments)

collection_map = {
    'iNat': 'birdclef-2026/train_audio',
    'XC': 'birdclef-2026/train_audio',
    'iNat2': 'birdclef-2025/train_audio',
    'XC2': 'birdclef-2025/train_audio',
    'CSA': 'birdclef-2025/train_audio',
    'esc': 'ESC-50-master/audio',
    'arca23k': 'ARCA23K/ARCA23K.audio/ARCA23K.audio',
    'urban8k': 'UrbanSound8K/audio/foldall',
    'dbr': 'dbr-dataset/dataset/dograin',
    'freesound': 'freesound/outside',
    'random-noise': 'random-noise',
}

if model_name is None:
    model_name = 'efficientnet_b0'
if num_epochs is None:
    num_epochs = 2
if batch_size is None:
    batch_size = 2 ** 7
if num_workers is None:
    num_workers = 0
if n_samples is None:
    n_samples = 1e9
if loss_name is None:
    loss_name = 'cross_entropy'
if optimizer_name is None:
    optimizer_name = 'adam'
if use_cosine_annealing is None:
    use_cosine_annealing = False
if dropout is None:
    dropout = 0.2
if lr is None:
    lr = 1e-3
if freeze_inner is None:
    freeze_inner = False
if output_folder is None:
    output_folder = '../results'
loss_folder = f"{output_folder}/loss"
metrics_folder = f"{output_folder}/metrics"

# trigger cli failure
assert table_path is not None, 'Must provide an audio metadata file'

### Importing packages and other setup

In [ ]:
# common imports
import os
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    precision_recall_fscore_support, 
    f1_score, 
    roc_auc_score
)

# modeling imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
from safetensors.torch import save_file
import json

# osprey package
from osprey import (
    AudioDataset,
    FocalBCEWithLogitsLoss,
    FocalCrossEntropyLoss,
    reformat_image,
    waveform_batch_to_mel,
 )

os.makedirs(output_folder, exist_ok=True)
os.makedirs(loss_folder, exist_ok=True)
os.makedirs(metrics_folder, exist_ok=True)

# Check if a GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU instead.")

### Loading and preparing the data

In [ ]:
# table with metadata about audio files
meta = pd.read_csv(table_path)

# subsets
train_data = meta[meta['dataset']=='train'].copy()
le = LabelEncoder() # label encoder
# le.fit(train_data['primary_label'])
le.fit(train_data['primary_label_append'])
val_data = meta[meta['dataset'].isin(['validate','test'])].copy()

# reset indices
train_data.reset_index(inplace=True, drop=True)
val_data.reset_index(inplace=True, drop=True)

# unique species in datasets
train_species = np.array(train_data.primary_label_append.unique())
val_species = np.array(val_data.primary_label_append.unique())

In [ ]:
# smaller subset: 8 batches of 32 samples = 256
num_classes = len(le.classes_)

# Determine label encoding based on loss function
# BCE-based losses (bce, focal_bce) require one-hot encoding
# CrossEntropy-based losses (cross_entropy, focal_cross_entropy) require class indices
encode_labels_onehot = loss_name in ['bce', 'focal_bce']

# sample from train/validation splits (use .head(n_samples) for deterministic order)
train_data_small = train_data.sample(
    n=min(n_samples, len(train_data)), 
    random_state=42).reset_index(drop=True)
val_data_small = val_data.sample(
    n=min(n_samples, len(val_data)), 
    random_state=42).reset_index(drop=True)

# Build waveform datasets; mel conversion + augmentation happen inside training/eval loops.
dataset_train = AudioDataset(
    train_data_small,
    le,
    base_folder=base_folder,
    collection_map=collection_map,
    sr=sr,
    duration=duration,
    encode_labels_onehot=encode_labels_onehot,
)
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, num_workers=num_workers, shuffle=False)

dataset_val = AudioDataset(
    val_data_small,
    le,
    base_folder=base_folder,
    collection_map=collection_map,
    sr=sr,
    duration=duration,
    encode_labels_onehot=encode_labels_onehot,
)
dataloader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=num_workers, shuffle=False)

print("Number of training samples:", len(train_data_small))
print("Number of batches:", len(dataloader_train))
print("Number of validation samples:", len(val_data_small))

### Model definition

In [ ]:
weights_map = {
    'efficientnet_b0': models.EfficientNet_B0_Weights.DEFAULT,
    'efficientnet_b1': models.EfficientNet_B1_Weights.DEFAULT,
    'efficientnet_b2': models.EfficientNet_B2_Weights.DEFAULT,
    'efficientnet_b3': models.EfficientNet_B3_Weights.DEFAULT,
    'mobilenet_v3_small': models.MobileNet_V3_Small_Weights.DEFAULT,
    'mobilenet_v3_large': models.MobileNet_V3_Large_Weights.DEFAULT,
    'regnet_y_400mf': models.RegNet_Y_400MF_Weights.DEFAULT,
    'regnet_y_8gf': models.RegNet_Y_8GF_Weights.DEFAULT,
    'regnet_x_400mf': models.RegNet_X_400MF_Weights.DEFAULT,
    'regnet_x_8gf': models.RegNet_X_8GF_Weights.DEFAULT,
}
weights = weights_map[model_name]
resize_size = (weights.transforms().resize_size)[0]
image_size = resize_size, resize_size
model_fn = getattr(models, model_name)
model = model_fn(weights=weights)

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=dropout, inplace=True),
    nn.Linear(in_features, num_classes),
)

# total parameter counts
total_params = sum(p.numel() for p in model.parameters())

if freeze_inner:

    # freeze all but the last layer
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

# trainable parameter counts
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

model.to(device)

# printing summary
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters: {total_params - trainable_params:,}")

### Optimizer and loss function

In [ ]:
if loss_name == 'bce':
    criterion = nn.BCEWithLogitsLoss()
elif loss_name == 'focal_bce':
    criterion = FocalBCEWithLogitsLoss(alpha=0.25, gamma=2.0)
elif loss_name == 'focal_cross_entropy':
    criterion = FocalCrossEntropyLoss(alpha=0.25, gamma=2.0)
elif loss_name == 'cross_entropy':
    criterion = nn.CrossEntropyLoss()
else:
    raise ValueError(f"Unknown loss_name: {loss_name}")

if optimizer_name == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=lr)
elif optimizer_name == 'adamw':
    optimizer = optim.AdamW(model.parameters(), lr=lr)
else:
    raise ValueError(f"Unknown optimizer_name: {optimizer_name}")

scheduler = None
if use_cosine_annealing:
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

### Training loop

In [ ]:
channel_size = 3

ft = open(f"{loss_folder}/training_loss.csv", 'w')
fv = open(f"{loss_folder}/validation_loss.csv", 'w')
ft.write('epoch,loss\n')
fv.write('epoch,loss\n')

for _ in range(1, num_epochs + 1):

    # learning
    model.train()
    loss_total = 0
    itr = 0
    for a in range(1, num_augments + 1):
        curr_time = time.time()
        itr = 0
        for X, Y in dataloader_train:
            itr += 1
            if (itr % 200) == 0:
                print(f'{itr/len(dataloader_train)*100:.2f}%')

            # Waveform augmentation then mel conversion happens inside the loop.
            mel_batch = waveform_batch_to_mel(
                X,
                sr=sr,
                fmin=fmin,
                fmax=fmax,
                duration=duration,
                apply_waveform_augment=True,
                p_augment=p_augment,
                min_gain_db=min_gain_db,
                max_gain_db=max_gain_db,
                mel_time_size=mel_time_size,
            )

            optimizer.zero_grad()
            Z = reformat_image(
                mel_batch, 
                image_size=image_size, 
                channel_size=channel_size,
            ).to(device)
            Y = Y.to(device)
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss
            loss.backward()
            optimizer.step()
        print(f'Epoch {_}, Round {a} took {time.time() - curr_time:0f} seconds')
    print('')

    # inference
    model.eval()
    with torch.no_grad():

        # training loss tracking (no waveform augmentation)
        loss_total = 0
        itr = 0
        for X, Y in dataloader_train:
            itr += 1
            mel_batch = waveform_batch_to_mel(
                X,
                sr=sr,
                fmin=fmin,
                fmax=fmax,
                duration=duration,
                apply_waveform_augment=False,
                mel_time_size=mel_time_size,
            )
            Z = reformat_image(
                mel_batch,
                image_size=image_size, 
                channel_size=channel_size,
            ).to(device)
            Y = Y.to(device)
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss
        print(f'The training loss at epoch {_} is {loss_total / itr:6f}')
        ft.write(f"{_},{loss_total / itr:4f}\n")

        # validation loss tracking (no waveform augmentation)
        loss_total = 0
        itr = 0
        for X, Y in dataloader_val:
            itr += 1
            mel_batch = waveform_batch_to_mel(
                X,
                sr=sr,
                fmin=fmin,
                fmax=fmax,
                duration=duration,
                apply_waveform_augment=False,
                mel_time_size=mel_time_size,
            )
            Z = reformat_image(
                mel_batch,
                image_size=image_size, 
                channel_size=channel_size,
            ).to(device)
            Y = Y.to(device)
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss
        print(f'The validation loss at epoch {_} is {loss_total / itr:6f}')
        fv.write(f"{_},{loss_total / itr:4f}\n")
    
    print('')

ft.close()
fv.close()

### Save model

In [ ]:
if model_path is not None:

    checkpoint_path = f"{output_folder}/{model_path}"

    tensor_state_dict = model.state_dict()

    metadata = {
        "optimizer_state_dict": json.dumps(optimizer.state_dict(), default=str),
        "num_classes": str(num_classes),
        "label_classes": json.dumps(le.classes_.tolist()),
        "image_size": json.dumps(image_size),
        "channel_size": str(channel_size),
        "epoch": str(num_epochs),
        "model_name": model_name,
    }

    save_file(tensor_state_dict, checkpoint_path, metadata=metadata)

    print(f"Checkpoint saved safely to: {checkpoint_path}")

### Evaluation loop

In [ ]:
# compute some summary metrics

zipped_data = [
    (train_species, dataloader_train, 'training'),
    (val_species, dataloader_val, 'validation'),
]

for species, dataloader, split_name in zipped_data:

    f0 = open(f"{metrics_folder}/{split_name}_metrics.csv",'w')
    
    loss_total = 0.0
    n_batches = 0

    model.eval()

    # Initialize empty lists to hold batch tensors
    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for X, Y_batch in dataloader:
            # Convert waveform batch to mel spectrograms inside the loop (no augmentation).
            mel_batch = waveform_batch_to_mel(
                X,
                sr=sr,
                fmin=fmin,
                fmax=fmax,
                duration=duration,
                apply_waveform_augment=False,
                mel_time_size=mel_time_size,
            )
            Z_batch = reformat_image(
                mel_batch,
                image_size=image_size,
                channel_size=channel_size,
            ).to(device)
            Y_batch = Y_batch.to(device)
            
            logits = model(Z_batch)
            loss = criterion(logits, Y_batch)

            loss_total += loss.item()
            n_batches += 1

            # Store the batch tensors directly
            y_true_all.append(Y_batch)
            y_pred_all.append(logits)

    # Concatenate all batches at once and move to CPU only once
    y_true = torch.cat(y_true_all).cpu().numpy()
    y_logits = torch.cat(y_pred_all).cpu().numpy()

    # Metrics depend on loss function
    if loss_name in ['bce', 'focal_bce']:
        # BCE multilabel: y_true is one-hot, calculate AUROC per label
        f0.write('label,auroc,threshold_auroc\n')
        
        # Calculate AUROC for each label
        y_probs = 1 / (1 + np.exp(-y_logits))  # sigmoid(logits)
        
        f0.write('overall,')
        try:
            # Macro AUROC across all labels
            overall_auroc = roc_auc_score(y_true, y_probs, average='macro')
            f0.write(f"{overall_auroc:.4f},0.5\n")
        except:
            f0.write("nan,0.5\n")
        
        # Per-label AUROC
        for label_idx in range(y_true.shape[1]):
            label_name = le.inverse_transform([label_idx])[0]
            try:
                auroc = roc_auc_score(y_true[:, label_idx], y_probs[:, label_idx])
                f0.write(f"{label_name},{auroc:.4f},0.5\n")
            except:
                f0.write(f"{label_name},nan,0.5\n")
    
    else:
        # CrossEntropy: y_true is class indices, calculate precision/recall/F1
        f0.write('category,loss,precision,recall,f1_score\n')
        
        y_pred = np.argmax(y_logits, axis=1)
        
        # write metrics
        f0.write('overall,')
        f0.write(f"{loss_total/n_batches:.2f},")
        f0.write(f"{precision_score(y_true, y_pred, average='macro', zero_division=0):.2f},")
        f0.write(f"{recall_score(y_true, y_pred, average='macro', zero_division=0):.2f},")
        f0.write(f"{f1_score(y_true, y_pred, average='macro', zero_division=0):.2f}")
        f0.write('\n')

        # metrics for every class
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred, average=None, zero_division=0
        )

        for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
            s = le.inverse_transform([i])[0]
            f0.write(f"{s},0.,{p:.2f},{r:.2f},{f:.2f}\n")

    f0.close()